# JJMO Flux Calibration Pipeline -- Quickstart Tutorial

This notebook demonstrates the full `jjmo_fluxcal` pipeline on JJMO Sirius data,
from raw segmented spectra to a flux-calibrated spectrum in physical units.

**Requirements:** Install the package first:
```bash
pip install -e .          # from the repository root
pip install -e ".[full]"  # includes optional dependencies (matplotlib, spectres, etc.)
```

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*Spectrum1D.*deprecated.*")
warnings.filterwarnings("ignore", category=DeprecationWarning)

import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

import jjmo_fluxcal
from jjmo_fluxcal import io, wavelength, quality, stitching, reference, sensitivity, calibrate
from jjmo_fluxcal.config import PipelineConfig

print(f"jjmo_fluxcal v{jjmo_fluxcal.__version__}")

## 1. Configuration

The pipeline is controlled by a `PipelineConfig` dataclass. You can use defaults,
override specific parameters, or load from a YAML file.

In [ ]:
# View defaults
cfg = PipelineConfig()
print(f"Star: {cfg.star_name}")
print(f"Fit method: {cfg.fit_method}, order: {cfg.fit_order}")
print(f"Sigma clip: {cfg.sigma_clip}")
print(f"Mask Balmer: {cfg.mask_balmer}, telluric: {cfg.mask_telluric}")

# Or load from YAML:
# cfg = PipelineConfig.from_yaml("jjmo_fluxcal/examples/sirius_config.yaml")

## 2. Step 1 -- Data Ingestion

Read all spectral segments from a directory. The module auto-detects
file formats (.fit/.txt pairs for Sirius, .csv for Betelgeuse).

In [ ]:
# Point to the Sirius data directory
SIRIUS_DIR = "/home/habjan.e/JJMO_home/Data/Sirius"

segments = io.read_directory(SIRIUS_DIR)
print(f"Loaded {len(segments)} segments")

for seg in segments:
    w = seg.spectral_axis.to(u.AA).value
    print(f"  {seg.meta.get('segment_id', '?'):>8s}: {w[0]:.0f} - {w[-1]:.0f} A  ({len(w)} px)")

In [ ]:
# Plot raw segments
fig, ax = plt.subplots(figsize=(12, 4))
for seg in segments:
    w = seg.spectral_axis.to(u.AA).value
    f = seg.flux.value
    ax.plot(w, f, lw=0.8, label=seg.meta.get("segment_id", ""))
ax.set_xlabel("Wavelength (A)")
ax.set_ylabel("Counts")
ax.set_title("Raw Sirius Segments")
ax.legend(fontsize=7, ncol=4)
plt.tight_layout()
plt.show()

## 3. Step 2 -- Wavelength Calibration

Detect absorption lines, match to known rest wavelengths, measure instrumental
offset (from telluric lines) and radial velocity (from stellar lines), then
correct wavelength arrays.

In [ ]:
# Extract arrays for wavelength calibration
wavelengths = [seg.spectral_axis.to(u.AA).value for seg in segments]
fluxes = [seg.flux.value for seg in segments]
seg_ids = [seg.meta.get("segment_id", f"seg_{i:02d}") for i, seg in enumerate(segments)]

# Run calibration
solutions = wavelength.calibrate_segments(wavelengths, fluxes, segment_ids=seg_ids)

# Print summary table
wavelength.print_calibration_table(solutions)

## 4. Step 3 -- Quality Assessment

Assess each segment for edge vignetting, cosmic rays, and SNR.
Build combined quality masks.

In [ ]:
# Use corrected wavelengths from Step 2
corrected_w = [sol.wavelength_corrected for sol in solutions]

reports = quality.assess_segments(corrected_w, fluxes, segment_ids=seg_ids)
quality.print_quality_table(reports)

## 5. Step 4 -- Stitching

Cross-normalize and stitch segments into a single continuous spectrum.

In [ ]:
# Build Spectrum1D list with corrected wavelengths
try:
    from specutils import Spectrum as Spectrum1D
except ImportError:
    from specutils import Spectrum1D

corrected_segments = []
for i, (w, f, qr) in enumerate(zip(corrected_w, fluxes, reports)):
    sp = Spectrum1D(
        spectral_axis=w * u.AA,
        flux=f * u.ct,
        mask=~qr.mask_good,  # specutils convention: True = bad
        meta={"segment_id": seg_ids[i]},
    )
    corrected_segments.append(sp)

# Stitch
result = stitching.stitch_segments(corrected_segments, normalize=True)

print(f"Stitched spectrum: {result.wavelength[0]:.0f} - {result.wavelength[-1]:.0f} A")
print(f"  {len(result.wavelength)} pixels")

In [ ]:
# Plot stitched spectrum
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(result.wavelength, result.flux, "k-", lw=0.5)
if result.uncertainty is not None:
    ax.fill_between(result.wavelength,
                     result.flux - result.uncertainty,
                     result.flux + result.uncertainty,
                     alpha=0.2, color="gray")
ax.set_xlabel("Wavelength (A)")
ax.set_ylabel("Counts")
ax.set_title("Stitched Sirius Spectrum (raw counts)")
plt.tight_layout()
plt.show()

## 6. Step 5 -- Reference Spectrum

Load the CALSPEC reference spectrum for Sirius. This provides the "true"
flux against which we derive the sensitivity function.

In [ ]:
# Load CALSPEC reference (requires network on first call; cached thereafter)
ref_spec = reference.load_reference_spectrum("sirius", prefer="calspec")

ref_w = ref_spec.spectral_axis.to(u.AA).value
ref_f = ref_spec.flux.value

print(f"Reference: {ref_w[0]:.0f} - {ref_w[-1]:.0f} A, {len(ref_w)} points")

## 7. Step 6 -- Sensitivity Function

Derive S(lambda) = F_ref / C_obs. This smooth function encodes the
instrument + atmosphere throughput.

In [ ]:
# Derive sensitivity function
sens_result = sensitivity.derive_sensitivity(
    corrected_segments,
    ref_spec,
    method="chebyshev",
    order=4,
    sigma_clip_threshold=3.0,
)

# derive_sensitivity may return dict or SensitivityFit
if isinstance(sens_result, dict):
    sens_fit = sens_result.get("global", sens_result.get("stitched"))
else:
    sens_fit = sens_result

print(f"Sensitivity function: {sens_fit.method}, order {sens_fit.order}")
print(f"  RMS residual: {sens_fit.rms_residual:.4f}")
print(f"  Points used: {sens_fit.n_points_used}, rejected: {sens_fit.n_rejected}")

## 8. Step 7 -- Flux Calibration

Apply the sensitivity function to produce the final calibrated spectrum
in physical flux units (erg/s/cm^2/A).

In [ ]:
# Apply calibration
cal_result = calibrate.apply_sensitivity(result, sens_fit)

print(f"Calibrated spectrum: {cal_result.wavelength[0]:.0f} - {cal_result.wavelength[-1]:.0f} A")
print(f"  Median flux: {np.nanmedian(cal_result.flux):.3e} erg/s/cm^2/A")

In [ ]:
# Final comparison: calibrated vs reference
fig, ax = plt.subplots(figsize=(12, 5))

# Calibrated spectrum
ax.plot(cal_result.wavelength, cal_result.flux, "k-", lw=0.6,
        label="Calibrated (JJMO)", alpha=0.8)
if cal_result.uncertainty is not None:
    ax.fill_between(cal_result.wavelength,
                     cal_result.flux - cal_result.uncertainty,
                     cal_result.flux + cal_result.uncertainty,
                     alpha=0.15, color="gray")

# Reference spectrum (trimmed to observed range)
wmin, wmax = cal_result.wavelength[0], cal_result.wavelength[-1]
mask_ref = (ref_w >= wmin) & (ref_w <= wmax)
ax.plot(ref_w[mask_ref], ref_f[mask_ref], "r-", lw=0.6,
        label="CALSPEC Reference", alpha=0.7)

ax.set_xlabel("Wavelength (A)")
ax.set_ylabel(r"Flux (erg s$^{-1}$ cm$^{-2}$ $\AA^{-1}$)")
ax.set_title("Sirius: Flux-Calibrated vs CALSPEC Reference")
ax.legend()
plt.tight_layout()
plt.show()

## 9. One-Call Pipeline

All the above steps can be run with a single function call:

In [ ]:
# One-call pipeline (uncomment to run):
#
# from jjmo_fluxcal import fluxcal
#
# result = fluxcal(
#     "/home/habjan.e/JJMO_home/Data/Sirius",
#     star_name="sirius",
#     output_dir="./results",
#     fit_order=4,
#     sigma_clip=3.0,
# )
#
# # result["calibrated"]  -- CalibrationResult
# # result["sensitivity"] -- SensitivityFit
# # result["stitched"]    -- StitchResult

print("Done! See result dict keys: calibrated, sensitivity, stitched, quality_reports, wavelength_solutions, config")

## 10. Command-Line Usage

The package also provides a CLI:

```bash
# Full pipeline
jjmo-fluxcal run --input-dir ./data/Sirius --star sirius --output-dir ./results

# Derive sensitivity function only
jjmo-fluxcal sensfunc --input-dir ./data/Sirius --star sirius --output sensfunc.json

# Apply saved sensitivity to new data
jjmo-fluxcal apply --sensfunc sensfunc.json --input science.fits --output calibrated.fits
```

Use `--config my_config.yaml` to load parameters from a YAML file.
Example configs are provided in `jjmo_fluxcal/examples/`.